## Práctica Procesado de Video: Sistema de Conteo Automático con Visión Artificial

#### Procesamiento del video

In [1]:
from ultralytics import YOLO

# 1. Cargar el modelo (usamos el nano 'n' por velocidad, ideal para video)
# Nota: Si 'yolo26n.pt' es hipotético, usa 'yolov8n.pt' o 'yolo11n.pt'
model = YOLO("yolo11n.pt")

# 2. Ejecutar la predicción sobre el archivo de video
# 'save=True' guarda el video resultante con las cajas dibujadas
# 'classes=[2, 3, 5, 7]' filtra para detectar solo vehículos (2: (Coche), 3: Motorcycle (Moto), 5: Bus (Autobús), 7: Truck (Camión))
results = model.predict(source="dataset/video_trafico.mp4", save=True, classes=[2, 3, 5, 7])
print("Video procesado y guardado.")


WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/9184) /home/ciabd10/Descargas/Modelos-Inteligencia-Artificial/Actividad_7/dataset/video_trafico.mp4: 384x640 7 cars, 171.7ms
video 1/1 (frame 2/9184) /home/ciabd10/Descargas/Modelos-Inteligencia-Artificial/Actividad_7/dataset/video_trafico.mp4: 384x640 7 cars, 4.8ms
video 1/1 (frame 3/9184) /home/ciabd10/Descargas/Modelos-Inteligencia-Artificial/Actividad_7/dataset/video_trafico.mp4: 384x640 8 cars, 4.8ms
video 1/1 (frame 4/9184) /ho

#### Procesamiento en Bucle con OpenCV

In [ ]:
import cv2
from ultralytics import YOLO
import sys

# 1. PARÁMETROS DE CALIBRACIÓN
# Estos valores dependen de la perspectiva de tu cámara
DISTANCIA_REFERENCIA_METROS = 10  
DISTANCIA_REFERENCIA_PIXELES = 400 
M_POR_PX = DISTANCIA_REFERENCIA_METROS / DISTANCIA_REFERENCIA_PIXELES

# 2. CARGAR MODELO
model = YOLO("yolo11n.pt")

# 3. CONFIGURACIÓN DE VIDEO
video_path = "dataset/video_trafico.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: No se pudo abrir el video.")
    sys.exit()

# Configurar el guardado del video de salida
ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
writer = cv2.VideoWriter('resultado/trafficCam_final.mp4',
                         cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho, alto))

# --- VARIABLES PARA SEGUIMIENTO ---
ids_contados = set()
posiciones_anteriores = {} # {id: (x, y)}
historial_velocidades = {} # {id: [lista de velocidades]}

# Diccionario de nombres de clases (car, truck, etc.)
nombres_clases = model.names

print("Procesando video: Mostrando Tipo y Velocidad (IDs ocultos)...")

# 4. BUCLE DE PROCESAMIENTO
frame_count = 0
while cap.isOpened():
    success, frame = cap.read()

    if success:
        # Aplicar tracking. persist=True mantiene el seguimiento del objeto.
        # Clases: 2: car, 3: motorcycle, 5: bus, 7: truck
        results = model.track(frame, conf=0.5, classes=[2, 3, 5, 7], persist=True, verbose=False)

        # 4.1. Lógica de cálculo (Cerebro)
        if results[0].boxes.id is not None:
            boxes_xywh = results[0].boxes.xywh.cpu().numpy() 
            ids = results[0].boxes.id.int().cpu().tolist()

            for box, id_vehiculo in zip(boxes_xywh, ids):
                ids_contados.add(id_vehiculo)
                x, y, w, h = box
                centro_actual = (x, y)

                if id_vehiculo in posiciones_anteriores:
                    pos_previa = posiciones_anteriores[id_vehiculo]
                    dist_px = ((centro_actual[0] - pos_previa[0])**2 + (centro_actual[1] - pos_previa[1])**2)**0.5
                    vel_kmh = (dist_px * M_POR_PX) * fps * 3.6

                    if id_vehiculo not in historial_velocidades:
                        historial_velocidades[id_vehiculo] = []
                    historial_velocidades[id_vehiculo].append(vel_kmh)
                    
                    if len(historial_velocidades[id_vehiculo]) > 5:
                        historial_velocidades[id_vehiculo].pop(0)
                
                posiciones_anteriores[id_vehiculo] = centro_actual

        # 4.2. RENDERIZADO (Lo que se ve)
        # labels=False oculta el texto predeterminado (Clase + ID) de YOLO
        frame_anotado = results[0].plot(labels=False)
        
        if results[0].boxes.id is not None:
            ids = results[0].boxes.id.int().cpu().tolist()
            clases = results[0].boxes.cls.int().cpu().tolist()
            coords = results[0].boxes.xyxy.cpu().numpy() # [x1, y1, x2, y2]

            for i, id_v in enumerate(ids):
                x1, y1, x2, y2 = coords[i]
                tipo_vehiculo = nombres_clases[clases[i]].upper()
                
                # Definir texto personalizado
                if id_v in historial_velocidades:
                    promedio_vel = sum(historial_velocidades[id_v]) / len(historial_velocidades[id_v])
                    texto = f"{tipo_vehiculo} {int(promedio_vel)} km/h"
                else:
                    texto = f"{tipo_vehiculo}"

                # Dibujar el texto sobre la caja con OpenCV
                # Color: Cian (0, 255, 255), Grosor: 2
                cv2.putText(frame_anotado, texto, (int(x1), int(y1) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        # Contador global en la esquina superior izquierda
        cv2.putText(frame_anotado, f"Vehiculos totales: {len(ids_contados)}", 
                    (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        # 4.3. GUARDAR Y PROGRESO
        writer.write(frame_anotado)
        frame_count += 1
        if frame_count % 30 == 0:
            print(f"Frames procesados: {frame_count} | Vehículos: {len(ids_contados)}", end="\r")
            
    else:
        break

# 5. FINALIZACIÓN
cap.release()
writer.release()
print(f"\n\nProceso finalizado.")
print(f"Resultado guardado en: resultado/trafficCam_final.mp4")

In [ ]:
## NOTAS:
# He añadido el cuentakilometros, pero el video al tener perspectiva, hace que la velocidad no sea precisa. 
# Para mejorar esto, se podría usar una calibración más avanzada o técnicas de visión por computadora para estimar la distancia real.